# Av Haritası — kuş fotoğrafı modeli (GPU eğitimi)

İstanbul'daki 333 kuş türü için fotoğraftan tür tanıma modeli. Model telefonda, internetsiz çalışır.

**Başlamadan:** üst menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**.

Hücreleri sırayla çalıştırın (▶). Bağlantı koparsa 1–3. hücreleri yeniden çalıştırın, sonra kaldığınız adımı.
Her adım kaldığı yerden devam eder; ilerleme Google Drive'da (`AvHaritasi_kus_foto`) saklanır.

| Adım | Süre (T4) |
|---|---|
| 4. Fotoğrafları indir | ~45–75 dk |
| 5. Eğit | ~3–4 saat |
| 6–8. Değerlendir, Core ML, yükle | ~10 dk |

Eğitim sürerken bu sekmeyi açık tutun.

## 1. GPU ve Google Drive

In [ ]:
!nvidia-smi -L || echo "GPU YOK: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU"
from google.colab import drive
drive.mount('/content/drive')

## 2. Kod ve kütüphaneler

In [ ]:
!test -d /content/Av && git -C /content/Av pull -q || git clone -q --depth 1 https://github.com/oguzbay-del/Av /content/Av
!pip install -q timm coremltools

## 3. Ayarlar

Varsayılanlar en iyi kalite için seçildi; değiştirmeniz gerekmez.

Eğitilen modelin otomatik yüklenmesi için (isteğe bağlı): soldaki 🔑 **Gizli anahtarlar** paneline
`HARITA_VERI_TOKEN` adıyla harita-veri deposuna yazma izni olan GitHub token'ı ekleyin ve
"Not defteri erişimi"ni açın. Eklemezseniz paket Drive'a kaydedilir.

In [ ]:
import os
os.environ.update({
    "MODEL": "tf_efficientnetv2_s.in21k_ft_in1k",  # ImageNet-21k ön eğitimli EfficientNetV2-S
    "IMG": "320",            # giriş boyutu (piksel)
    "EPOCHS": "20",          # eğitim turu
    "PER_SPECIES": "400",    # tür başına en çok fotoğraf
    "MIN_PHOTOS": "25",      # bundan az fotoğrafı olan tür eğitime alınmaz
    "LICENSES": "cc0,cc-by,cc-by-nc",  # uygulama ücretli olacaksa: "cc0,cc-by"
    "VERI": "/content/veri",
    "YEDEK": "/content/drive/MyDrive/AvHaritasi_kus_foto",
    "DEPO": "/content/Av",
})
try:
    from google.colab import userdata
    os.environ["GH_TOKEN"] = userdata.get("HARITA_VERI_TOKEN")
    print("GitHub token bulundu: paket otomatik yüklenecek")
except Exception:
    print("GitHub token yok: paket Drive'a kaydedilecek")
K = "python /content/Av/tools/kus_foto/kus_foto.py"

## 4. Fotoğrafları indir (iNaturalist, serbest lisanslı)

In [ ]:
!{K} indir

## 5. Eğit

Her turun sonunda doğrulama doğruluğu yazılır ve kontrol noktası Drive'a kaydedilir.

In [ ]:
!{K} egit

## 6. Değerlendir

Test fotoğraflarında doğruluk ve en önemlisi: **avlanamayan kuşun av türü sanılma oranı**.

In [ ]:
!{K} degerlendir

## 7. Core ML'e dönüştür

In [ ]:
!{K} aktar

## 8. Paketle ve yükle

Bitince çıktıdaki sürüm etiketini (ör. `kus-foto-20261011-0930`) ya da Drive'daki `BirdClassifier_paket.zip` dosyasını iletin.

In [ ]:
!{K} yukle